# S5-nano (Maritaca AI) no ENEM 2026

O S5-nano abriu os pesos em 07/11/2026; as provas do ENEM 2026 foram aplicadas em 08/11 e 15/11/2026, então o modelo nunca viu essas questões.
O modelo recebe as **páginas do caderno azul como imagem** e responde as 90 questões do dia escolhido.

- 1º dia: Linguagens (01-45, nas questões 01 a 05 a opção de inglês) e Ciências Humanas (46-90).
- 2º dia: Ciências da Natureza (91-135) e Matemática (136-180).
- Prova e gabarito: **oficiais do INEP** (caderno azul). Questões anuladas ficam de fora da conta.

**Antes de rodar:** Ambiente de execução → Alterar o tipo de ambiente → **GPU** (T4 ou L4). Escolha o dia em `DIA` na primeira célula e use Ambiente de execução → Executar tudo.

In [ ]:
!pip -q install -U transformers accelerate pymupdf pytesseract
!apt-get -qq install -y tesseract-ocr tesseract-ocr-por > /dev/null

DIA = 2  # 1 ou 2
CADERNO = {1: "D1_CD1", 2: "D2_CD7"}[DIA]  # caderno azul de cada dia
# Cópia dos PDFs oficiais do INEP no repositório: download.inep.gov.br não responde a partir do Colab
INEP = "https://raw.githubusercontent.com/YasRordrigues/s5nano-enem-2026/main/provas/2026_{}_impresso_" + CADERNO + ".pdf"
!curl -sSfL -A "Mozilla/5.0" -o prova.pdf {INEP.format("PV")}
!curl -sSfL -A "Mozilla/5.0" -o gabarito.pdf {INEP.format("GB")}

import re, json, time, os
import pymupdf, pytesseract, torch
from PIL import Image

QUESTOES = range(1, 91) if DIA == 1 else range(91, 181)
AREAS = {1: [("Linguagens", 1, 45), ("Humanas", 46, 90)],
         2: [("Natureza", 91, 135), ("Matemática", 136, 180)]}[DIA]
def area(n):
    return next(a for a, i, f in AREAS if i <= n <= f)

# Gabarito oficial do INEP. No 1º dia a primeira letra de 01-05 é a de inglês; anuladas ficam como None
txt = " ".join(p.get_text() for p in pymupdf.open("gabarito.pdf"))
GAB = {}
for m in re.finditer(r"\b(\d{1,3})\s+(Anulad[oa]|[A-E])\b", txt):
    n = int(m.group(1))
    if n in QUESTOES and n not in GAB:
        GAB[n] = None if m.group(2).startswith("Anulad") else m.group(2)
assert len(GAB) == 90, len(GAB)
VALIDAS = [n for n in QUESTOES if GAB[n]]
print(f"Gabarito oficial: {len(VALIDAS)} questões válidas, anuladas: {[n for n in QUESTOES if not GAB[n]]}")

# Páginas em PNG (150 dpi) e em que página e posição aparece cada "QUESTÃO NN":
# primeiro pelo texto do PDF, depois por OCR (cabeçalhos que estão como imagem)
doc = pymupdf.open("prova.pdf")
os.makedirs("paginas", exist_ok=True)
heads = {}  # n -> (pagina, coluna, y em pontos)
for p in range(doc.page_count):
    pg = doc[p]
    fn = f"paginas/p{p:02d}.png"
    pg.get_pixmap(dpi=150).save(fn)
    achadas = []
    ws = pg.get_text("words")
    for i, w in enumerate(ws[:-1]):
        m = re.fullmatch(r"0?(\d{1,3})", ws[i + 1][4])
        if re.fullmatch(r"QUEST\S?O", w[4]) and m:
            achadas.append((int(m.group(1)), int(w[0] > pg.rect.width / 2), w[1]))
    img = Image.open(fn)
    d = pytesseract.image_to_data(img, lang="por", output_type=pytesseract.Output.DICT)
    for i, w in enumerate(d["text"]):
        m = re.match(r"\s*0?(\d{1,3})\b", " ".join(d["text"][i + 1:i + 3]))
        if re.fullmatch(r"QUEST\S?O", w.strip()) and m:
            achadas.append((int(m.group(1)), int(d["left"][i] > img.width / 2), d["top"][i] * 72 / 150))
    for n, col, y in achadas:
        if n in QUESTOES and n not in heads:  # 1ª vez = inglês em 01-05
            heads[n] = (p, col, y)
faltando = [n for n in QUESTOES if n not in heads]
print(f"{doc.page_count} páginas, {len(heads)}/90 questões localizadas. Faltando: {faltando}")

# Questões não localizadas ficam com as páginas das vizinhas
def paginas_da(n):
    if n in heads:
        p = heads[n][0]
        mesma = sorted((c, y, k) for k, (pp, c, y) in heads.items() if pp == p)
        ultima = mesma[-1][2] == n
        return [p, p + 1] if ultima and p + 1 < doc.page_count else [p]
    antes = max((heads[k][0] for k in heads if k < n), default=1)
    depois = min((heads[k][0] for k in heads if k > n), default=doc.page_count - 1)
    return list(range(antes, depois + 1))

PAGS = {n: paginas_da(n) for n in QUESTOES}
print({n: PAGS[n] for n in list(QUESTOES)[::15]})

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL = "maritaca-ai/S5-nano"
# A T4 não tem bf16 nativo; se as respostas saírem estranhas no teste rápido, troque para torch.bfloat16
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForMultimodalLM.from_pretrained(MODEL, dtype=DTYPE, device_map="auto")
print(torch.cuda.get_device_name(), DTYPE)

PROMPT = ("Você está fazendo a prova do ENEM 2026. As imagens são páginas do caderno de questões. "
          "Responda à QUESTÃO {n}{lingua}. Explique o raciocínio em poucas linhas e termine com uma linha "
          "no formato 'Resposta: X', em que X é A, B, C, D ou E.")

def responder(n, pensar=False, max_new_tokens=1024):
    lingua = " (a versão de inglês, não a de espanhol)" if n <= 5 else ""
    content = [{"type": "image", "path": f"paginas/p{p:02d}.png"} for p in PAGS[n]]
    content.append({"type": "text", "text": PROMPT.format(n=f"{n:02d}", lingua=lingua)})
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", enable_thinking=pensar).to(model.device)
    sampling = dict(temperature=1.0, top_p=0.95) if pensar else dict(temperature=0.7, top_p=0.8)
    torch.manual_seed(0)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_k=20, **sampling)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    texto = processor.decode(gen, skip_special_tokens=True)
    final = texto.split("</think>")[-1]
    m = re.findall(r"Resposta:\s*\**\(?([A-E])\b", final)
    letra = m[-1] if m else None
    return {"questao": n, "area": area(n), "paginas": PAGS[n],
            "gabarito": GAB[n], "resposta": letra, "certa": bool(letra and letra == GAB[n]),
            "segundos": round(time.time() - t0, 1), "tokens": len(gen), "texto": texto[-1500:]}

def rodar(questoes, pensar=False, max_new_tokens=1024):
    res = []
    for n in questoes:
        r = responder(n, pensar, max_new_tokens)
        res.append(r)
        print(f"Q{n:02d} gabarito {r['gabarito']} resposta {r['resposta']} {'✔' if r['certa'] else '✘'} "
              f"({r['segundos']} s, {r['tokens']} tokens)", flush=True)
    return res

def resumo(res, nome):
    ok = sum(r["certa"] for r in res)
    print(f"\n{nome}: {ok}/{len(res)} ({100 * ok / len(res):.1f}%)")
    for a, _, _ in AREAS:
        rs = [r for r in res if r["area"] == a]
        if rs:
            print(f"  {a}: {sum(r['certa'] for r in rs)}/{len(rs)}")
    print(f"  sem resposta: {sum(r['resposta'] is None for r in res)}, "
          f"média {sum(r['segundos'] for r in res) / len(res):.1f} s por questão")

In [ ]:
# Teste rápido: 3 questões. Confira se o texto faz sentido antes de rodar tudo.
teste = rodar([VALIDAS[0], VALIDAS[45], VALIDAS[-1]])
print(teste[0]["texto"])

In [ ]:
# Modo Instruct (resposta direta), todas as questões válidas do dia. Na L4 leva por volta de 30 a 60 minutos.
# No 1º dia o limite foi 1024 tokens; no 2º, 2048, porque Matemática pede contas mais longas.
MAX_TOKENS = 1024 if DIA == 1 else 2048
instruct = rodar(VALIDAS, pensar=False, max_new_tokens=MAX_TOKENS)
resumo(instruct, f"S5-nano Instruct, {DIA}º dia")
print("COMPACTO", json.dumps({r["questao"]: [r["resposta"], int(r["certa"]), r["segundos"]] for r in instruct}))
json.dump({"modelo": MODEL, "modo": "instruct", "dia": DIA, "gabarito": "oficial INEP", "max_new_tokens": MAX_TOKENS,
           "resultados": instruct},
          open(f"resultado_s5nano_enem2026_d{DIA}_instruct.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Salva no seu Google Drive (pasta linkedin-resultados)
from google.colab import drive
import shutil
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/linkedin-resultados', exist_ok=True)
for f in os.listdir('.'):
    if f.startswith('resultado_s5nano'):
        shutil.copy(f, '/content/drive/MyDrive/linkedin-resultados/' + f)
        print('Salvo em Meu Drive/linkedin-resultados/' + f)

### Opcional: modo Think

In [ ]:
# Opcional, se sobrar tempo: modo Think (raciocina antes), que é bem mais lento.
# Comece com LIMITE = 15 para ver o tempo por questão; use None para todas.
LIMITE = 15
qs = VALIDAS[:LIMITE] if LIMITE else VALIDAS
think = rodar(qs, pensar=True, max_new_tokens=8192)
resumo(think, f"S5-nano Think, {DIA}º dia")
print("COMPACTO", json.dumps({r["questao"]: [r["resposta"], int(r["certa"]), r["segundos"]] for r in think}))
json.dump({"modelo": MODEL, "modo": "think", "dia": DIA, "gabarito": "oficial INEP", "resultados": think},
          open(f"resultado_s5nano_enem2026_d{DIA}_think.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Salva no seu Google Drive (pasta linkedin-resultados)
from google.colab import drive
import shutil
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/linkedin-resultados', exist_ok=True)
for f in os.listdir('.'):
    if f.startswith('resultado_s5nano'):
        shutil.copy(f, '/content/drive/MyDrive/linkedin-resultados/' + f)
        print('Salvo em Meu Drive/linkedin-resultados/' + f)